In [ ]:
#!/usr/bin/env python3
import pickle, requests

In [ ]:
personal_access_token = "d11a7c908f48-c37279e1e259dee96efee4344494fca8a4c1bf2286d3223419e452318c1723b966782e3a71e26550f790d99a5870d609a6690bc3359f5e19ed7a12fa3740f6d8"
cookies_file = "cookies.pickle"

In [ ]:
with requests.Session() as s:
    with open(cookies_file, 'rb') as f:
            s.cookies.update(pickle.load(f))
            print(f)

In [ ]:
# uff, this might become hell to handle this api...but better than an iffy interface still?!

In [ ]:
with requests.Session() as s:
    # Load cookies
    try:
        with open(cookies_file, 'rb') as f:
            s.cookies.update(pickle.load(f))
    except FileNotFoundError:
        print("No cookies file detected!")
 
    # Log in
    r = s.post('https://korpus.cz/login', data={'personal_access_token': personal_access_token})
 
    # Creating a concordance query
    request_body = {
        "type": "concQueryArgs",
        "maincorp": "syn2015",
        "usesubcorp": None,
        "viewmode": "kwic",
        "pagesize": 40,
        "attrs": ["word","tag"],
        "attr_vmode": "visible-kwic",
        "base_viewattr": "word",
        "ctxattrs": [],
        "structs": ["text","p","g"],
        "refs": [],
        "fromp": 0,
        "shuffle": 0,
        "queries": [
            {
                "qtype": "advanced",
                "corpname": "syn2015",
                "query": "[word=\"celou\"] [lemma=\"pravda\"]",
                "pcq_pos_neg": "pos",
                "include_empty": False,
                "default_attr": "word"
            }
        ],
        "text_types": {},
        "context":
        {
            "fc_lemword_wsize": [-5, 5],
            "fc_lemword": "",
            "fc_lemword_type": "all",
            "fc_pos_wsize": [-5, 5],
            "fc_pos": [],
            "fc_pos_type": "all"
        },
        "async": False
    }
    r = s.post('https://korpus.cz/kontext-api/v0.17/query_submit', params={'format': 'json'}, json=request_body)
    response_json = r.json()
    print(response_json)
 
    # Displaying a concordance
    conc_persistence_op_id = response_json['conc_persistence_op_id']
    r = s.get('https://korpus.cz/kontext-api/v0.17/view', params={'format': 'json', 'q': '~'+conc_persistence_op_id})
    print(r.json())
 
    # Store cookies
    with open(cookies_file, 'wb') as f:
        pickle.dump(s.cookies, f)

